# Prompt-injection guardrail benchmark — fine-tune on Colab

Runtime → Change runtime type → **T4 GPU** before running.

This notebook: clones the repo, pulls the dataset, fine-tunes DeBERTa-v3-small, evaluates it with the same harness as the off-the-shelf detectors, and (optionally) pushes the model to your Hugging Face account.

In [ ]:
!nvidia-smi -L

## 1. Clone the repo and install

In [ ]:
REPO_URL = 'https://github.com/Alexander-Maldonado-Pelayo/prompt-injection-guardrail-bench'  # <- edit if different
!git clone $REPO_URL
%cd prompt-injection-guardrail-bench
!pip install -q -r requirements.txt

## 2. Log in to Hugging Face
Needed to download Prompt Guard 2 (gated) and to push your model. Create a **write** token at https://huggingface.co/settings/tokens.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 3. Pull the dataset (English rows) and look at the profile

In [ ]:
!python -m data.load
from IPython.display import Markdown
Markdown(open('data/profile.md').read())

## 4. Off-the-shelf baselines on the test split
The Ollama-based detectors (Llama Guard 3, Qwen3 zero-shot) run on your own machine, not here.

In [ ]:
!python -m eval.run --predictor heuristic protectai promptguard2 --split test

## 5. Fine-tune
~5 epochs on ~1.5K English rows takes a few minutes on a T4. Early stopping picks the best epoch by validation F1.

In [ ]:
!python -m train.finetune --base microsoft/deberta-v3-small --epochs 5

## 6. Evaluate the fine-tuned model with the same harness

In [ ]:
!python -m eval.run --predictor finetuned --model-id ./train/out/best --split test

## 7. Table + chart

In [ ]:
!python -m eval.report --split test
from IPython.display import Image
Image('results/recall_vs_fpr__test.png')

## 8. Error analysis — read these and write the findings section

In [ ]:
!python -m eval.errors results/*__test.json
!head -40 results/errors__protectai-v2__test.md

## 9. (Optional) push the model to the Hub and commit results back to GitHub

In [ ]:
# HF_REPO = 'xRainbowdartx/deberta-injection-hardneg'
# !python -m train.finetune --base microsoft/deberta-v3-small --epochs 5 --push $HF_REPO

# Download the results folder to commit locally:
# !zip -r results.zip results && from google.colab import files; files.download('results.zip')